# 01A — Telecom Pack

**Outcome:** translate the native telecom files into the small, standardized
pack interface consumed by Notebook 01B.

This notebook owns telecom meaning: native file names, ONTs, network
relationships, metric units, and telecom fault labels. It does **not** create
canonical long telemetry and it does not contain modelling features.


## 1. Setup

Keep this notebook and `week1_core.py` together in
`MyDrive/anomaly_detection/research/week1/`.

The default run uses the complete observable panel. For a quick check, set
`TELECOM_ENTITY_IDS`, `TELECOM_SAMPLE_START`, or `TELECOM_SAMPLE_END` before
running the notebook. Completed pack directories are immutable; change
`TELECOM_PACK_RUN_ID` before rerunning.


In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "week1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from week1_core import (
    EVAL_SCHEMAS,
    PACK_METRIC_SCHEMA,
    finalise_pack,
    immutable_directory,
    pack_core_hashes,
    read_json,
    sha256_file,
    source_record,
)

SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT",
    DRIVE_ROOT / "telco_syntetic_data",
))
PACK_RUN_ID = os.getenv(
    "TELECOM_PACK_RUN_ID",
    "telecom_v4_1_full_v1",
)
PACK_ROOT = (
    DRIVE_ROOT / "outputs" / "packs" / "telecom" / PACK_RUN_ID
)
BATCH_ROWS = int(os.getenv("TELECOM_BATCH_ROWS", "100000"))
ENTITY_IDS = tuple(filter(None, os.getenv(
    "TELECOM_ENTITY_IDS", ""
).split(",")))
SAMPLE_START = os.getenv("TELECOM_SAMPLE_START") or None
SAMPLE_END = os.getenv("TELECOM_SAMPLE_END") or None
RUN_BUILD = os.getenv("RUN_TELECOM_PACK", "1") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "entity_filter": ENTITY_IDS or "all",
    "sample_start": SAMPLE_START or "first timestamp",
    "sample_end": SAMPLE_END or "last timestamp",
    "batch_rows": BATCH_ROWS,
}, name="value").to_frame())


## 2. The Telecom phrasebook

`native_field` is used only while reading the telecom panel. `metric_id` is the
standardized name written to the pack. The remaining columns tell the common
adapter how to preserve units, measurement behaviour, and source censoring.

`clip_at=5_000_000` records the generator's FEC ceiling. The common adapter
will mark values at that ceiling as `clipped`; it will not change the value.


In [ ]:
METRIC_COLUMNS = ["native_field", *PACK_METRIC_SCHEMA]
metric_map = pd.DataFrame([
    ("rx_power_dbm", "rx_power_dbm", "ont", "gauge", "dBm", None),
    ("olt_rx_power_dbm", "olt_rx_power_dbm", "ont", "gauge", "dBm", None),
    ("tx_power_dbm", "tx_power_dbm", "ont", "gauge", "dBm", None),
    ("temperature_c", "temperature_c", "ont", "gauge", "degC", None),
    ("bias_current_ma", "bias_current_ma", "ont", "gauge", "mA", None),
    ("voltage_v", "voltage_v", "ont", "gauge", "V", None),
    ("ber", "ber", "ont", "bounded_fraction", "ratio", None),
    ("fec_count", "fec_count", "ont", "interval_count", "count", 5_000_000),
    ("crc_errors", "crc_errors", "ont", "interval_count", "count", None),
    ("uptime_s", "uptime_s", "ont", "cumulative_counter", "s", None),
    ("reboot_count", "reboot_count", "ont", "cumulative_counter", "count", None),
    ("throughput_mbps", "throughput_mbps", "ont", "gauge", "Mbps", None),
], columns=METRIC_COLUMNS)

relation_map = pd.DataFrame([
    ("olt_id", "pon_port", "contains"),
    ("pon_port", "splitter_l1", "contains"),
    ("splitter_l1", "splitter_l2", "contains"),
    ("splitter_l2", "ont_id", "serves"),
    ("geo_cluster", "ont_id", "groups"),
], columns=["parent_field", "child_field", "relation_type"])

display(metric_map)
display(relation_map)


## 3. Inspect the native source

The observable panel may be Parquet or CSV. Parquet is preferred for the full
dataset. The check below deliberately fails if any `gt_*`, `class`, or `state`
field is still present in the observable panel.

`tickets.csv` is explicitly classified as evaluation input.


In [ ]:
def source_roots(root, evaluation=False):
    roots = [root, root / "Data", root / "native"]
    if evaluation:
        roots += [
            root / "evaluation",
            root / "SPEC-EVAL",
            root / "SPEC_EVAL",
        ]
    return roots


def find_file(root, name, evaluation=False, required=True):
    matches = [
        folder / name
        for folder in source_roots(root, evaluation)
        if (folder / name).is_file()
    ]
    if len(matches) > 1:
        raise ValueError(f"Ambiguous source file {name}: {matches}")
    if matches:
        return matches[0]
    if required:
        raise FileNotFoundError(f"Missing source file: {name}")
    return None


def find_panel(root):
    for name in ("reference_dataset.parquet", "reference_dataset.csv"):
        path = find_file(root, name, required=False)
        if path is not None:
            return path
    candidates = []
    for folder in source_roots(root):
        candidates += sorted(folder.glob("reference_dataset*.parquet"))
    if not candidates:
        for folder in source_roots(root):
            candidates += sorted(folder.glob("reference_dataset*.csv"))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one reference_dataset file, found {candidates}"
        )
    return candidates[0]


def panel_columns(path):
    if path.suffix == ".parquet":
        return pq.ParquetFile(path).schema_arrow.names
    return pd.read_csv(path, nrows=0).columns.tolist()


def iter_panel(path, columns, batch_rows):
    if path.suffix == ".parquet":
        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(
            batch_size=batch_rows,
            columns=columns,
        ):
            yield batch.to_pandas()
    else:
        yield from pd.read_csv(
            path,
            usecols=columns,
            chunksize=batch_rows,
        )


panel_path = find_panel(SOURCE)
native_columns = panel_columns(panel_path)
truth_columns = sorted(
    name for name in native_columns
    if name.startswith("gt_") or name in {"class", "state"}
)
available_metrics = metric_map.loc[
    metric_map["native_field"].isin(native_columns)
].copy()

inventory = {
    "panel": panel_path.name,
    "topology": find_file(SOURCE, "topology.csv").name,
    "service_windows": find_file(
        SOURCE, "entity_service_windows.csv"
    ).name,
    "fault_registry": find_file(
        SOURCE, "gt_fault_registry.csv", evaluation=True
    ).name,
    "fault_intervals": find_file(
        SOURCE, "fault_entity_intervals.csv", evaluation=True
    ).name,
    "tickets": (
        find_file(
            SOURCE, "tickets.csv", evaluation=True, required=False
        ) is not None
    ),
    "observable_truth_columns": truth_columns,
    "mapped_metrics": len(available_metrics),
}
display(pd.Series(inventory, name="value").to_frame())

assert {"timestamp_utc", "ont_id"} <= set(native_columns)
assert not truth_columns, (
    "Use the updated observable reference dataset; truth must not be in it."
)
assert not available_metrics.empty


## 4. Build the standardized Telecom Pack

The output is still wide and efficient:

```text
PACK-CORE/
  observations/part-*.parquet
  metric_catalogue.parquet
  entity_registry.parquet
  entity_relations.parquet
PACK-EVAL/
  fault_events.parquet
  fault_entity_intervals.parquet
  tickets.parquet
```

Only this notebook knows how those tables are obtained from telecom files.


In [ ]:
def select_rows(frame, entity_ids=(), start=None, end=None):
    selected = frame.copy()
    if entity_ids:
        selected = selected.loc[
            selected["ont_id"].astype(str).isin(entity_ids)
        ]
    timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if start:
        selected = selected.loc[
            timestamps.ge(pd.to_datetime(start, utc=True))
        ]
        timestamps = pd.to_datetime(selected["timestamp_utc"], utc=True)
    if end:
        selected = selected.loc[
            timestamps.lt(pd.to_datetime(end, utc=True))
        ]
    return selected.reset_index(drop=True)


def pick_column(frame, *names):
    for name in names:
        if name in frame:
            return frame[name]
    return pd.Series(pd.NA, index=frame.index)


def telecom_registry(topology, windows, data_start, data_end):
    entity_fields = [
        ("olt_id", "olt"),
        ("pon_port", "pon_port"),
        ("splitter_l1", "splitter_l1"),
        ("splitter_l2", "splitter_l2"),
        ("geo_cluster", "geo_cluster"),
        ("ont_id", "ont"),
    ]
    windows = windows.copy()
    windows["entity_id"] = windows["entity_id"].astype(str)
    windows = windows.set_index("entity_id")
    rows = []
    for field, entity_type in entity_fields:
        for entity_id in sorted(
            topology[field].dropna().astype(str).unique()
        ):
            valid_from, valid_to = data_start, data_end
            if entity_type == "ont" and entity_id in windows.index:
                window = windows.loc[entity_id]
                valid_from = pd.to_datetime(
                    window["install_ts"], utc=True, errors="coerce"
                )
                valid_to = pd.to_datetime(
                    window["decommission_ts"], utc=True, errors="coerce"
                )
            rows.append((entity_id, entity_type, valid_from, valid_to))
    return pd.DataFrame(
        rows,
        columns=["entity_id", "entity_type", "valid_from", "valid_to"],
    ).drop_duplicates("entity_id")


def telecom_relations(topology):
    rows = []
    for parent_field, child_field, relation_type in relation_map.itertuples(
        index=False
    ):
        pairs = topology[[parent_field, child_field]].dropna().drop_duplicates()
        rows.extend(
            (str(parent), str(child), relation_type)
            for parent, child in pairs.itertuples(index=False)
        )
    return pd.DataFrame(
        rows,
        columns=[
            "parent_entity_id",
            "child_entity_id",
            "relation_type",
        ],
    ).drop_duplicates()


def telecom_evaluation(root, selected_entities):
    registry = pd.read_csv(find_file(
        root, "gt_fault_registry.csv", evaluation=True
    ))
    intervals = pd.read_csv(find_file(
        root, "fault_entity_intervals.csv", evaluation=True
    ))
    intervals = intervals.loc[
        intervals["entity_id"].astype(str).isin(selected_entities)
    ].copy()
    selected_faults = set(intervals["fault_id"].dropna().astype(str))
    registry_id = pick_column(registry, "gt_fault_id", "fault_id").astype(
        "string"
    )
    registry = registry.loc[registry_id.isin(selected_faults)].copy()

    fault_events = pd.DataFrame({
        "fault_id": pick_column(
            registry, "gt_fault_id", "fault_id"
        ).astype("string"),
        "fault_type": pick_column(
            registry, "gt_fault_type", "fault_type"
        ).astype("string"),
        "domain_id": pick_column(registry, "target", "domain_id").astype(
            "string"
        ),
        "onset_ts": pick_column(registry, "onset_ts"),
        "observable_ts": pick_column(
            registry, "first_observable_ts", "observable_ts"
        ),
        "impact_ts": pick_column(registry, "impact_ts"),
        "end_ts": pick_column(registry, "repair_ts", "end_ts"),
        "group_id": pick_column(registry, "group_id").astype("string"),
    })
    fault_intervals = pd.DataFrame({
        "fault_id": intervals["fault_id"].astype("string"),
        "entity_id": intervals["entity_id"].astype("string"),
        "start_ts": pick_column(
            intervals, "active_start_ts", "start_ts"
        ),
        "end_ts": pick_column(intervals, "active_end_ts", "end_ts"),
    })

    ticket_path = find_file(
        root, "tickets.csv", evaluation=True, required=False
    )
    if ticket_path is None:
        tickets = pd.DataFrame(columns=EVAL_SCHEMAS["tickets"])
    else:
        native_tickets = pd.read_csv(ticket_path)
        native_tickets = native_tickets.loc[
            native_tickets["ont_id"].astype(str).isin(selected_entities)
        ]
        tickets = pd.DataFrame({
            "ticket_id": native_tickets["ticket_id"].astype("string"),
            "entity_id": native_tickets["ont_id"].astype("string"),
            "reported_ts": pick_column(native_tickets, "reported_ts"),
            "resolved_ts": pick_column(native_tickets, "resolved_ts"),
            "fault_id": pick_column(
                native_tickets, "gt_fault_id", "fault_id"
            ).astype("string"),
        })

    tables = {
        "fault_events": fault_events,
        "fault_entity_intervals": fault_intervals,
        "tickets": tickets,
    }
    for name, frame in tables.items():
        for column in frame:
            if column.endswith("_ts"):
                frame[column] = pd.to_datetime(
                    frame[column], utc=True, errors="coerce"
                )
        tables[name] = frame[EVAL_SCHEMAS[name]]
    return tables


In [ ]:
def build_telecom_pack(
    source,
    destination,
    *,
    include_evaluation=True,
    entity_ids=(),
    start=None,
    end=None,
    batch_rows=100_000,
):
    source, destination = Path(source), Path(destination)
    panel = find_panel(source)
    columns = panel_columns(panel)
    catalogue = metric_map.loc[
        metric_map["native_field"].isin(columns)
    ].copy()
    read_columns = [
        "timestamp_utc",
        "ont_id",
        *catalogue["native_field"],
    ]
    rename_metrics = dict(zip(
        catalogue["native_field"], catalogue["metric_id"]
    ))

    with immutable_directory(destination) as pack:
        core = pack / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)

        presence_parts = []
        part_number = 0
        for batch in iter_panel(panel, read_columns, batch_rows):
            batch = select_rows(batch, entity_ids, start, end)
            if batch.empty:
                continue
            wide = batch.rename(columns={
                "timestamp_utc": "event_ts",
                "ont_id": "entity_id",
                **rename_metrics,
            })
            wide["event_ts"] = pd.to_datetime(wide["event_ts"], utc=True)
            wide["entity_id"] = wide["entity_id"].astype(str)
            wide = wide[
                ["event_ts", "entity_id", *catalogue["metric_id"]]
            ]
            wide.to_parquet(
                observations / f"part-{part_number:05d}.parquet",
                index=False,
                compression="zstd",
            )
            presence_parts.append(
                wide[["event_ts", "entity_id"]].drop_duplicates()
            )
            part_number += 1

        if not presence_parts:
            raise ValueError("The selected telemetry slice is empty")
        presence = pd.concat(presence_parts, ignore_index=True)
        selected_entities = set(presence["entity_id"])
        data_start = presence["event_ts"].min()
        data_end = presence["event_ts"].max() + pd.Timedelta(minutes=15)

        topology_path = find_file(source, "topology.csv")
        windows_path = find_file(source, "entity_service_windows.csv")
        topology = pd.read_csv(topology_path)
        topology = topology.loc[
            topology["ont_id"].astype(str).isin(selected_entities)
        ].copy()
        windows = pd.read_csv(windows_path)
        windows = windows.loc[
            windows["entity_id"].astype(str).isin(selected_entities)
        ].copy()

        catalogue[PACK_METRIC_SCHEMA].to_parquet(
            core / "metric_catalogue.parquet", index=False
        )
        telecom_registry(
            topology, windows, data_start, data_end
        ).to_parquet(core / "entity_registry.parquet", index=False)
        telecom_relations(topology).to_parquet(
            core / "entity_relations.parquet", index=False
        )

        evaluation_tables = []
        if include_evaluation:
            evaluation = pack / "PACK-EVAL"
            evaluation.mkdir()
            tables = telecom_evaluation(source, selected_entities)
            evaluation_tables = list(tables)
            for name, frame in tables.items():
                frame.to_parquet(
                    evaluation / f"{name}.parquet", index=False
                )

        source_files = [
            source_record(panel, source),
            source_record(topology_path, source),
            source_record(windows_path, source),
        ]
        if include_evaluation:
            for name in (
                "gt_fault_registry.csv",
                "fault_entity_intervals.csv",
                "tickets.csv",
            ):
                path = find_file(
                    source, name, evaluation=True, required=False
                )
                if path is not None:
                    source_files.append(
                        source_record(path, source, evaluation_only=True)
                    )

        finalise_pack(
            pack,
            sector="telecom",
            pack_version="0.1.0",
            expected_cadence_seconds=900,
            source_manifest={
                "source_id": "telemetry-synth-4.1.0",
                "source_root": str(source),
                "files": source_files,
                "selection": {
                    "entity_ids": list(entity_ids),
                    "start": start,
                    "end": end,
                },
                "evaluation_files_never_read_for_pack_core": True,
            },
            evaluation_tables=evaluation_tables,
            notes=[
                "FEC values at 5000000 are source-censored.",
                "tickets.csv is evaluation-only.",
                "geographic membership is retained as a non-tree relation.",
            ],
        )

    return read_json(destination / "pack_manifest.json")


if RUN_BUILD:
    pack_manifest = build_telecom_pack(
        SOURCE,
        PACK_ROOT,
        include_evaluation=True,
        entity_ids=ENTITY_IDS,
        start=SAMPLE_START,
        end=SAMPLE_END,
        batch_rows=BATCH_ROWS,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(
    pack_manifest["core_row_counts"], name="rows"
).to_frame())
display(pd.Series(
    pack_manifest["evaluation_row_counts"], name="rows"
).to_frame())


## 5. Prove the Telecom Pack does not leak truth

This is the primary isolation test because the sector notebook is the only
component that can see native observations and native truth together.

Two small native fixtures are created:

1. original — observable inputs plus fault files and `tickets.csv`;
2. redacted — the same observable rows, with evaluation files absent.

Both are translated independently. Their logical `PACK-CORE` hashes must be
identical. A deliberately leaky signature that reads tickets must differ,
which proves the test can catch the relevant leak.


In [ ]:
def make_telecom_isolation_fixtures(source, original, redacted):
    original.mkdir()
    redacted.mkdir()

    panel = find_panel(source)
    columns = panel_columns(panel)
    safe_columns = [
        "timestamp_utc",
        "ont_id",
        *metric_map.loc[
            metric_map["native_field"].isin(columns), "native_field"
        ],
    ]
    sample = next(iter_panel(panel, safe_columns, 5_000))
    sample_entities = sorted(sample["ont_id"].astype(str).unique())[:2]
    sample = sample.loc[
        sample["ont_id"].astype(str).isin(sample_entities)
    ].copy()

    topology = pd.read_csv(find_file(source, "topology.csv"))
    safe_topology_columns = sorted(set(
        relation_map["parent_field"]
    ) | set(relation_map["child_field"]))
    topology = topology.loc[
        topology["ont_id"].astype(str).isin(sample_entities),
        safe_topology_columns,
    ]
    windows = pd.read_csv(
        find_file(source, "entity_service_windows.csv")
    )
    windows = windows.loc[
        windows["entity_id"].astype(str).isin(sample_entities),
        ["entity_id", "install_ts", "decommission_ts"],
    ]

    for fixture in (original, redacted):
        sample.to_parquet(
            fixture / "reference_dataset.parquet", index=False
        )
        topology.to_csv(fixture / "topology.csv", index=False)
        windows.to_csv(
            fixture / "entity_service_windows.csv", index=False
        )

    for name in (
        "gt_fault_registry.csv",
        "fault_entity_intervals.csv",
        "tickets.csv",
    ):
        path = find_file(
            source, name, evaluation=True, required=False
        )
        if path is not None:
            shutil.copy2(path, original / name)


def deliberately_leaky_signature(source):
    tickets = find_file(
        source, "tickets.csv", evaluation=True, required=False
    )
    return "missing" if tickets is None else sha256_file(tickets)


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    original_source = temporary / "native_original"
    redacted_source = temporary / "native_redacted"
    make_telecom_isolation_fixtures(
        SOURCE, original_source, redacted_source
    )

    original_pack = temporary / "pack_original"
    redacted_pack = temporary / "pack_redacted"
    build_telecom_pack(
        original_source,
        original_pack,
        include_evaluation=True,
        batch_rows=2_000,
    )
    build_telecom_pack(
        redacted_source,
        redacted_pack,
        include_evaluation=False,
        batch_rows=2_000,
    )

    original_hashes = pack_core_hashes(original_pack)
    redacted_hashes = pack_core_hashes(redacted_pack)
    assert original_hashes == redacted_hashes
    assert (
        deliberately_leaky_signature(original_source)
        != deliberately_leaky_signature(redacted_source)
    )

print("PASS — PACK-CORE is invariant after truth and tickets are removed")
print("PASS — the negative control detects the removed evaluation input")


## 6. Handoff to the common adapter

Notebook 01A is complete when:

- `PACK-CORE` contains only observable standardized data;
- `PACK-EVAL` is physically separate;
- the two isolation assertions pass.

Next, open `01B_COMMON_CANONICAL_ADAPTER.ipynb` and set
`ADAPTER_PACK_ROOT` to the printed pack path.


In [ ]:
summary = {
    "pack_root": str(PACK_ROOT),
    "sector": pack_manifest["sector"],
    "pack_version": pack_manifest["pack_version"],
    "metrics": len(pack_manifest["metric_ids"]),
    "pack_core_hashes": pack_manifest["core_content_hashes"],
    "next_notebook": "01B_COMMON_CANONICAL_ADAPTER.ipynb",
}
display(pd.Series(summary, name="value").to_frame())
